## Step 1: Housekeeping - Import Libraries and Set File Paths

In [17]:
import sqlite3
import csv
import os 

## Step 2: Download CSV Files

In [18]:
BOOK_PATH = "Book.csv"
MEMBER_PATH = "Member.csv"
LOAN_PATH = "Loan.csv"
DB_PATH = "library.db"

print("Book exists:", os.path.exists(BOOK_PATH))
print("Member exists:", os.path.exists(MEMBER_PATH))
print("Loan exists:", os.path.exists(LOAN_PATH))

Book exists: True
Member exists: True
Loan exists: True


## Step 3: Create Database and Tables

In [19]:
conn = sqlite3.connect(DB_PATH)
conn.execute("PRAGMA foreign_keys = ON;")

conn.execute("""
CREATE TABLE IF NOT EXISTS Book (
    callNo TEXT NOT NULL,
    title TEXT NOT NULL,
    author TEXT NOT NULL,
    PRIMARY KEY (callNo)
);
""")

conn.execute("""
CREATE TABLE IF NOT EXISTS Member (
    id INTEGER NOT NULL,
    firstname TEXT NOT NULL,
    lastName TEXT NOT NULL,
    PRIMARY KEY (id)
);
""")

conn.execute("""
CREATE TABLE IF NOT EXISTS Loan (
    callNo TEXT NOT NULL,
    id INTEGER NOT NULL,
    dateBorrowed TEXT NOT NULL,
    dateReturned TEXT,
    dateDue TEXT NOT NULL,
    PRIMARY KEY (callNo, id, dateBorrowed),
    FOREIGN KEY (callNo) REFERENCES Book(callNo),
    FOREIGN KEY (id) REFERENCES Member(id)
);
""")

conn.commit()
print("Tables created.")

Tables created.


## Verify Tables

In [20]:
conn.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;").fetchall()

[('Book',), ('Loan',), ('Member',)]

## Step 4: Load Book Data


In [21]:
with open(BOOK_PATH, newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        conn.execute(
            "INSERT INTO Book (callNo, title, author) VALUES (?, ?, ?);",
            (row["callNo"], row["title"], row["author"])
        )

conn.commit()
print("Book rows loaded:", conn.execute("SELECT COUNT(*) FROM Book;").fetchone()[0])

IntegrityError: UNIQUE constraint failed: Book.callNo

## Step 5: Load Member Data

In [ ]:
with open(MEMBER_PATH, newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        conn.execute(
            "INSERT INTO Member (id, firstname, lastName) VALUES (?, ?, ?);",
            (int(row["id"]), row["firstname"], row["lastName"])
        )

conn.commit()
print("Member rows loaded:", conn.execute("SELECT COUNT(*) FROM Member;").fetchone()[0])

Member rows loaded: 4


## Step 6: Load Loan Data

In [ ]:
with open(LOAN_PATH, newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        date_returned = row["dateReturned"] if row["dateReturned"].strip() else None

        conn.execute(
            """INSERT INTO Loan (callNo, id, dateBorrowed, dateReturned, dateDue)
               VALUES (?, ?, ?, ?, ?);""",
            (row["callNo"], int(row["id"]), row["dateBorrowed"], date_returned, row["dateDue"])
        )

conn.commit()
print("Loan rows loaded:", conn.execute("SELECT COUNT(*) FROM Loan;").fetchone()[0])

Loan rows loaded: 4


## Final Row Counts

In [ ]:
print("Book:", conn.execute("SELECT COUNT(*) FROM Book;").fetchone()[0])
print("Member:", conn.execute("SELECT COUNT(*) FROM Member;").fetchone()[0])
print("Loan:", conn.execute("SELECT COUNT(*) FROM Loan;").fetchone()[0])

Book: 11
Member: 4
Loan: 4


## Query 1: Retrieve all columns from the Book table ordered by author

In [ ]:
query1 = """
SELECT *
FROM Book
ORDER BY author;
"""

for row in conn.execute(query1):
    print(row)

('R 487 T35 1967', 'Medicine in medieval England.', 'Charles H Talbot')
('QA 76.9 D26H39 1996', 'Data model patterns : conventions of thought', 'David Hay')
('CB 351 M293 1983', 'Atlas of medieval Europe', 'Donald Matthew')
('HQ 1143 P68 1975', 'Medieval women', 'Eileen Power')
('PC 14 V48 1965', 'Medieval miscellany', 'Frederick Whitehead')
('QA 76.73 S67C435 2004', "Joe Celko's Trees and hierarchies in SQL for smarties", 'Joe Celko')
('QA 76.73 S67C46 1997', "Joe Celko's SQL puzzles & answers", 'Joe Celko')
('QA 76.9 D35C45 1999', "Joe Celko's data & databases : concepts in practice", 'Joe Celko')
('R 141 E45 2006', 'Medieval medicine and the plague', 'Lynne Elliott')
('QA 76.9 D26H355 2008', 'Information modeling and relational databases', 'T A Halpin')
('QA 76.76 A65P76 2011', 'Programming Android', 'Zigurd R Mednieks')


## Query 2: Retrieve the title of each book and the first and last name of the member who borrowed it, for all loans where the book has not yet been returned

In [ ]:
query2 = """
SELECT Book.title, Member.firstname, Member.lastName
FROM Loan
JOIN Book ON Loan.callNo = Book.callNo
JOIN Member ON Loan.id = Member.id
WHERE Loan.dateReturned IS NULL;
"""

for row in conn.execute(query2):
    print(row)

("Joe Celko's SQL puzzles & answers", 'David', 'Martin')
('Medieval medicine and the plague', 'David', 'Martin')


## Query 3: Retrieve the full loan history for the book with call number R 141 E45 2006

In [ ]:
query3 = """
SELECT Member.firstname, Member.lastName, Loan.dateBorrowed, Loan.dateDue, Loan.dateReturned
FROM Loan
JOIN Member ON Loan.id = Member.id
WHERE Loan.callNo = 'R 141 E45 2006'
ORDER BY Loan.dateBorrowed ASC;
"""

for row in conn.execute(query3):
    print(row)

('Betty', 'Freeman', '4/1/2014 0:00', '4/15/2014 0:00', '4/15/2014 0:00')
('David', 'Martin', '4/30/2014 0:00', '5/14/2014 0:00', None)


## Query 4: Retrieve the id, firstname, and lastName of every member who does not appear in the Loan table

In [ ]:
query4 = """
SELECT Member.id, Member.firstname, Member.lastName
FROM Member
LEFT JOIN Loan ON Member.id = Loan.id
WHERE Loan.id IS NULL;
"""

for row in conn.execute(query4):
    print(row)

(4, 'John', 'Martin')


## Query 5: Retrieve each member's full name and total number of loans, including members with zero loans

In [ ]:
query5 = """
SELECT Member.firstname, Member.lastName, COUNT(Loan.id) AS total_loans
FROM Member
LEFT JOIN Loan ON Member.id = Loan.id
GROUP BY Member.id, Member.firstname, Member.lastName
ORDER BY total_loans DESC;
"""

for row in conn.execute(query5):
    print(row)

('David', 'Martin', 2)
('John', 'Smith', 1)
('Betty', 'Freeman', 1)
('John', 'Martin', 0)


## Query 6: Which members currently have books checked out?
This is useful because the library can identify members with outstanding loans and see how many books each one still has.

In [ ]:
query6 = """
SELECT Member.firstname, Member.lastName, COUNT(Loan.callNo) AS books_outstanding
FROM Member
JOIN Loan ON Member.id = Loan.id
WHERE Loan.dateReturned IS NULL
GROUP BY Member.id, Member.firstname, Member.lastName;
"""

for row in conn.execute(query6):
    print(row)

('David', 'Martin', 2)


## Summary

The `dateReturned` column may be empty, in which case it must be converted to NULL when the data is imported into SQLite. This is one issue with the data quality of this dataset. This dataset is too small and basic for a real library system because it lacks multiple book copies, fines, reservations, and member contact details. This database is useful for practice, but more columns and tables would be needed for real-world use.


In [ ]:
conn.close()
print("Database connection closed.")

Database connection closed.
